# Systems, Linux, and Networking Foundations

Maps to `design3.md` Phase 3.

This notebook is a guided walkthrough of the operating system and network layers that sit beneath your application code. The goal is to remove black-box thinking: when a pod gets OOMKilled, when you see "too many open files," when a request hangs, or when a race condition corrupts shared state, you should be able to reason about why and where.

The standard here is practical understanding, not academic completeness. Every topic ties back to the kind of systems you actually operate: Kafka consumers, data pipelines, K8s pods, database connections, and API integrations.

Working rule:

- read the explanation first
- run the code cell
- modify one example before moving on
- write one short note in your own words after each section

## Learning Goals

By the end of this notebook, you should be able to:

- explain the difference between processes, threads, and coroutines, and choose the right one for a given workload
- describe how virtual memory, the stack, and the heap work at a level sufficient to debug OOM kills and stack overflows
- explain what file descriptors are, why they leak, and how to prevent it
- reason about buffered I/O and flush semantics for data durability
- write concurrent code using threading with locks, multiprocessing, and asyncio
- implement a producer-consumer pipeline
- trace a network request from DNS through TCP through TLS to HTTP response
- diagnose which network layer failed given symptoms
- use basic Linux debugging tools to inspect process, file, and network state

---

## 1. Processes vs Threads vs Coroutines

These are the three fundamental units of concurrent execution, and choosing the wrong one is a common source of performance bugs and unnecessary complexity.

### Process

A process is an independent instance of a running program. The operating system gives each process its own virtual address space, file descriptor table, and execution context. Processes are created by the OS via `fork` (Unix) or `spawn` (Windows/cross-platform). They are the heaviest unit of execution but provide true isolation and true parallelism.

Key properties:

- separate memory space -- one process cannot accidentally corrupt another's data
- truly parallel on multi-core hardware, even in CPython
- expensive to create (milliseconds) and expensive to communicate between (data must be serialized)
- each K8s pod container runs as a process; each worker in a Gunicorn deployment is a process

### Thread

A thread is a lightweight execution context within a process. All threads in a process share the same memory space, file descriptors, and heap. Threads are created by the OS but are cheaper than processes.

Key properties:

- shared memory makes communication easy but introduces race conditions
- in CPython, the GIL means only one thread executes Python bytecode at a time
- threads are still useful for I/O-bound work because waiting on I/O releases the GIL
- typical use: database connection pools, concurrent file operations, background logging
- creation cost is moderate (microseconds to low milliseconds)

### Coroutine

A coroutine is a function that can suspend and resume cooperatively within a single thread. In Python, coroutines are created with `async def` and yield control with `await`. They are managed by an event loop, not by the OS scheduler.

Key properties:

- lightest unit: creation cost is negligible (nanoseconds)
- no OS thread overhead, so you can run tens of thousands concurrently
- cooperative: a coroutine must explicitly yield; a blocking call freezes the entire event loop
- ideal for high-concurrency I/O: thousands of API calls, websocket connections, database queries
- async web frameworks (FastAPI, aiohttp) are built on this model

### When to use each

| Workload | Best fit | Why |
|----------|----------|-----|
| CPU-bound Python (number crunching, feature computation) | Process | Bypasses GIL, real parallelism |
| I/O-bound with shared state (DB connections, file ops) | Thread | Shared memory, GIL released during I/O |
| High-concurrency I/O (thousands of connections, API fan-out) | Coroutine | Minimal overhead per connection |
| Simple script, no concurrency needed | Sequential | No overhead, easiest to debug |

### Cost comparison

```
Process creation:  ~1-10 ms,  ~10+ MB memory overhead
Thread creation:   ~0.1 ms,   ~1 MB stack per thread (configurable)
Coroutine creation: ~0.001 ms, ~1 KB overhead
```

### Industry context

Your K8s pods run as processes -- that is why they have independent memory and crash independently. Database connection pools often use threads internally because each connection blocks on network I/O. FastAPI uses coroutines to handle thousands of concurrent HTTP requests without thousands of threads. Kafka consumers in your energy trading platform each run as a process (one consumer per pod), but within each consumer, async I/O handles concurrent offset commits and health checks.

In [ ]:
import math
import threading
import time
from concurrent.futures import ThreadPoolExecutor

# --- CPU-bound task: compute-heavy work ---
def cpu_work(n: int) -> float:
    """Simulate CPU-bound work: sum of square roots."""
    return sum(math.sqrt(i) for i in range(n))

# --- I/O-bound task: simulate network wait ---
def io_work(seconds: float) -> float:
    """Simulate I/O-bound work: sleep."""
    time.sleep(seconds)
    return seconds

WORK_SIZE = 500_000
IO_DELAY = 0.15
NUM_TASKS = 6

# --- Sequential baseline for CPU-bound ---
start = time.perf_counter()
cpu_results_seq = [cpu_work(WORK_SIZE) for _ in range(NUM_TASKS)]
cpu_sequential = time.perf_counter() - start

# --- Threaded CPU-bound (GIL limits this) ---
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=NUM_TASKS) as pool:
    cpu_results_thr = list(pool.map(lambda _: cpu_work(WORK_SIZE), range(NUM_TASKS)))
cpu_threaded = time.perf_counter() - start

# --- Sequential baseline for I/O-bound ---
start = time.perf_counter()
io_results_seq = [io_work(IO_DELAY) for _ in range(NUM_TASKS)]
io_sequential = time.perf_counter() - start

# --- Threaded I/O-bound (threads shine here) ---
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=NUM_TASKS) as pool:
    io_results_thr = list(pool.map(lambda _: io_work(IO_DELAY), range(NUM_TASKS)))
io_threaded = time.perf_counter() - start

print("CPU-bound comparison:")
print(f"  Sequential: {cpu_sequential:.3f}s")
print(f"  Threaded:   {cpu_threaded:.3f}s  (GIL prevents real speedup)")
print()
print("I/O-bound comparison:")
print(f"  Sequential: {io_sequential:.3f}s")
print(f"  Threaded:   {io_threaded:.3f}s  (threads overlap the waits)")
print(f"  Speedup:    {io_sequential / io_threaded:.1f}x")

# Try next:
# 1. Increase NUM_TASKS to 20 and observe how I/O speedup changes.
# 2. Reduce WORK_SIZE and see if threaded CPU-bound gets closer to sequential.
# 3. Think about what would happen if you used processes instead of threads for CPU-bound.

---

## 2. Virtual Memory, Stack, and Heap

Understanding memory layout is essential for diagnosing OOMKilled pods, stack overflows, and unexpected memory growth in long-running pipeline processes.

### Virtual memory

Every process gets its own virtual address space. The OS maps virtual addresses to physical RAM using page tables. This means:

- two processes can use the same virtual address but point to different physical memory
- the OS can give each process the illusion of having a large contiguous address space
- physical RAM is shared and managed transparently by the kernel

### Pages and page faults

Memory is managed in fixed-size chunks called pages (typically 4 KB). When your code accesses a virtual address:

1. The CPU checks the page table for a mapping to physical RAM.
2. If the page is in RAM, the access proceeds (fast).
3. If the page is NOT in RAM (it was never loaded, or it was swapped to disk), the CPU raises a **page fault**.
4. The OS handles the fault by loading the page from disk into RAM (slow -- microseconds to milliseconds).

This is why memory-mapped files and swap can cause surprising latency spikes. Your code looks like a normal memory access, but underneath it triggers disk I/O.

### Stack

Each thread gets its own stack, which stores:

- function call frames (local variables, return addresses)
- grows and shrinks automatically as functions are called and return
- has a fixed maximum size (typically 1-8 MB per thread)

When the stack exceeds its limit, you get a **stack overflow**. In Python, this manifests as `RecursionError` because Python enforces its own recursion limit before the OS stack limit is hit.

### Heap

The heap is where dynamically allocated objects live. In Python, essentially every object (dicts, lists, strings, class instances) is allocated on the heap. The heap:

- grows dynamically as objects are created
- is managed by Python's memory allocator and garbage collector
- has no fixed size limit (bounded only by available virtual memory)
- fragmentation can cause the process to hold more memory than the live objects need

### The OOM killer

When Linux runs out of memory (physical RAM + swap), the kernel's OOM killer selects a process to terminate. It picks the process using the most memory that is not critical to the system. This is why your K8s pods get `OOMKilled`:

- the pod's memory limit is hit
- the kernel kills the main process in the container
- K8s reports the exit code 137 (128 + SIGKILL signal 9)

Common causes in data pipelines:

- loading an entire large dataset into memory instead of streaming
- unbounded caches or dictionaries that grow with input size
- memory leaks from objects that are referenced but never cleaned up

### Industry context

When you see `OOMKilled` on a Kafka consumer pod, the first question is: what is accumulating in memory? Is the consumer buffering too many unprocessed messages? Is a deduplication set growing without bounds? Understanding that all Python objects live on the heap, and the heap grows until the OS kills you, is the mental model that drives the debugging.

In [ ]:
import sys

# --- Recursion limit: Python's guard against stack overflow ---
print(f"Default recursion limit: {sys.getrecursionlimit()}")

def recursive_depth(n: int) -> int:
    """Count how deep we can recurse before hitting the limit."""
    if n <= 0:
        return 0
    return 1 + recursive_depth(n - 1)

# Safe recursion
safe_result = recursive_depth(100)
print(f"Recursive depth (100): {safe_result}")

# Demonstrate RecursionError
try:
    recursive_depth(sys.getrecursionlimit() + 100)
except RecursionError as exc:
    print(f"RecursionError hit: {type(exc).__name__}")

# --- Memory usage of objects on the heap ---
small_list = [1, 2, 3]
large_list = list(range(100_000))
small_dict = {"a": 1}
large_dict = {i: i for i in range(100_000)}

print()
print("Object sizes (shallow, bytes):")
print(f"  small_list (3 items):       {sys.getsizeof(small_list):>10}")
print(f"  large_list (100k items):    {sys.getsizeof(large_list):>10}")
print(f"  small_dict (1 item):        {sys.getsizeof(small_dict):>10}")
print(f"  large_dict (100k items):    {sys.getsizeof(large_dict):>10}")
print()
print("Note: sys.getsizeof() shows the container's own size, not the size of")
print("the objects it contains. A list of dicts is much larger than getsizeof shows.")

# Try next:
# 1. Set sys.setrecursionlimit(200) and see how deep you can go.
# 2. Create a deeply nested dict and measure its shallow vs deep memory footprint.
# 3. Think about what happens to your Kafka consumer if it accumulates 10M dedup keys.

---

## 3. File Descriptors

In Unix, "everything is a file." File descriptors (FDs) are the integer handles that a process uses to interact with open files, sockets, pipes, and devices. Understanding them is essential for debugging "too many open files" errors in long-running services.

### What they are

When a process opens a file, socket, or pipe, the kernel assigns a small non-negative integer -- the file descriptor. The process uses this integer for all subsequent read/write/close operations. Every process starts with three:

- FD 0: standard input (stdin)
- FD 1: standard output (stdout)
- FD 2: standard error (stderr)

### The FD table and its limits

Each process has a file descriptor table with a configurable maximum size:

- default soft limit: often 1024 (can be raised with `ulimit -n`)
- hard limit: set by the system administrator
- in K8s pods, this is configurable via security context or init containers

### The leak pattern

The most common FD leak in data pipelines:

1. Open a file or socket.
2. An exception occurs before the close call.
3. The FD stays open.
4. Repeat thousands of times over hours or days.
5. Eventually: `OSError: [Errno 24] Too many open files`

This is insidious because it works fine in testing (short-lived processes) and only fails in production (long-running services).

### The fix: context managers

The `with` statement guarantees cleanup even if an exception occurs:

```python
# BAD: leak if an exception occurs between open and close
f = open("data.csv")
data = f.read()
f.close()

# GOOD: guaranteed cleanup
with open("data.csv") as f:
    data = f.read()
```

This applies to database connections, HTTP sessions, temporary files -- anything that holds a resource.

### Debugging tools

- `lsof -p PID`: list all open files for a process
- `/proc/PID/fd/`: directory containing symlinks to all open FDs (Linux only)
- `ls -la /proc/self/fd/`: see your own process's open FDs

### Industry context

A Kafka consumer that opens connections to multiple brokers and to TimescaleDB, combined with temporary files for batch processing, can easily approach FD limits if connections are not properly pooled and closed. When you see "too many open files" in a pod log, the debugging path is: `lsof -p PID | wc -l` to see how many FDs are open, then `lsof -p PID | sort` to see what they are (sockets? files? pipes?).

In [ ]:
import os
import tempfile

# --- Demonstrate FD leak and fix ---

# Show current open FD count (works on Windows and Linux)
def count_open_fds() -> int:
    """Count open file descriptors for this process (cross-platform approximation)."""
    count = 0
    for fd in range(1024):
        try:
            os.fstat(fd)
            count += 1
        except OSError:
            pass
    return count

baseline_fds = count_open_fds()
print(f"Baseline open FDs: {baseline_fds}")

# --- BAD: leaking file descriptors ---
leaked_files = []
temp_dir = tempfile.mkdtemp()
for i in range(50):
    path = os.path.join(temp_dir, f"leak_{i}.txt")
    f = open(path, "w")
    f.write(f"data {i}")
    leaked_files.append(f)
    # Notice: we never close f!

after_leak = count_open_fds()
print(f"After leaking 50 files: {after_leak} FDs open (+{after_leak - baseline_fds})")

# Clean up the leak
for f in leaked_files:
    f.close()

after_cleanup = count_open_fds()
print(f"After closing leaked files: {after_cleanup} FDs open")

# --- GOOD: context manager guarantees cleanup ---
for i in range(50):
    path = os.path.join(temp_dir, f"safe_{i}.txt")
    with open(path, "w") as f:
        f.write(f"data {i}")
    # f is guaranteed closed here, even if an exception occurred

after_safe = count_open_fds()
print(f"After context-managed writes: {after_safe} FDs open (no leak)")

# Clean up temp dir
import shutil
shutil.rmtree(temp_dir)

# Try next:
# 1. Remove the cleanup loop and see FDs stay leaked until process exits.
# 2. On Linux, run: ls -la /proc/{os.getpid()}/fd/ to see the actual FD entries.
# 3. Think about what happens to a Kafka consumer that opens broker connections without closing them.

---

## 4. Buffered I/O and Flush Semantics

When your code writes data, it does not go directly to disk. There are multiple layers of buffering between your `write()` call and the data being durable on physical storage. Understanding this path is critical for data pipeline durability.

### The write path

```
Your Python write() call
    |
    v
Python's userspace buffer (in-memory, managed by the file object)
    |  <-- flush() pushes data from here to the kernel
    v
Kernel page cache (in-memory, managed by the OS)
    |  <-- fsync() pushes data from here to physical storage
    v
Physical disk / SSD
```

### Userspace buffers

Python's file objects buffer writes in memory for efficiency. By default:

- text mode files are line-buffered (when connected to a terminal) or block-buffered (when connected to a file)
- binary mode files are block-buffered (typically 8 KB blocks)
- `flush()` forces the Python buffer to the kernel, but does NOT guarantee data is on disk

### Kernel page cache

The OS kernel maintains its own cache of disk pages in RAM. When Python flushes to the kernel, the data sits in the page cache. The kernel writes it to disk later, asynchronously, for performance. This means:

- if the process crashes after `flush()` but before the kernel writes to disk, your data is usually safe (the kernel still has it)
- if the **machine** crashes (power loss, kernel panic), data in the page cache is lost

### fsync: the durability guarantee

`os.fsync(fd)` forces the kernel to write cached data to physical storage. This is expensive (can take milliseconds) but is the only way to guarantee durability against machine crashes.

### Why this matters for data pipelines

A pipeline stage that "writes" trade events to a file or database WAL has not actually persisted them until the data reaches stable storage. If you report "successfully processed" before fsync, a crash can lose data silently.

This is exactly why:

- databases use write-ahead logs (WAL) with fsync on commit
- Kafka brokers fsync producer acknowledgments (configurable with `acks` setting)
- log rotation can lose the most recent entries if the process crashes before flush

### Industry context

When your Kafka producer uses `acks=all`, the broker guarantees the message has been replicated and fsynced. When you use `acks=1`, the leader has the data in its page cache but may not have fsynced. The tradeoff is latency vs durability -- the same tradeoff this buffering model creates everywhere.

In [ ]:
import os
import tempfile
import time

# --- Demonstrate buffered vs unbuffered writes ---

temp_dir = tempfile.mkdtemp()
buffered_path = os.path.join(temp_dir, "buffered.txt")
unbuffered_path = os.path.join(temp_dir, "unbuffered.txt")

NUM_WRITES = 5_000
DATA = "trade_event|EURUSD|1.0845|100|2026-04-04T08:15:00Z\n"

# --- Buffered write (default) ---
start = time.perf_counter()
with open(buffered_path, "w") as f:
    for _ in range(NUM_WRITES):
        f.write(DATA)
    # Data is flushed automatically when the file is closed by the context manager
buffered_time = time.perf_counter() - start

# --- Unbuffered write (flush after every write) ---
start = time.perf_counter()
with open(unbuffered_path, "w") as f:
    for _ in range(NUM_WRITES):
        f.write(DATA)
        f.flush()  # Force Python buffer to kernel after every write
unbuffered_time = time.perf_counter() - start

# --- fsync write (flush + fsync after every write) ---
fsynced_path = os.path.join(temp_dir, "fsynced.txt")
start = time.perf_counter()
with open(fsynced_path, "w") as f:
    for _ in range(NUM_WRITES):
        f.write(DATA)
        f.flush()
        os.fsync(f.fileno())  # Force kernel cache to physical disk
fsynced_time = time.perf_counter() - start

print(f"Buffered (default):    {buffered_time:.4f}s")
print(f"Flush after each:      {unbuffered_time:.4f}s  ({unbuffered_time/buffered_time:.1f}x slower)")
print(f"Flush + fsync each:    {fsynced_time:.4f}s  ({fsynced_time/buffered_time:.1f}x slower)")
print()
print("Takeaway: buffering exists for a reason. fsync is the most expensive")
print("because it waits for physical disk confirmation. Use it only when you")
print("need durability guarantees (e.g., WAL commits, critical checkpoints).")

# --- Show that unflushed data can be invisible ---
check_path = os.path.join(temp_dir, "check.txt")
f = open(check_path, "w")
f.write("important data that has not been flushed yet")
# At this point, another process reading check.txt might see an empty file
disk_size = os.path.getsize(check_path)
print(f"\nFile size before flush: {disk_size} bytes (data still in Python buffer)")
f.flush()
disk_size_after = os.path.getsize(check_path)
print(f"File size after flush:  {disk_size_after} bytes (data now in kernel cache)")
f.close()

# Clean up
import shutil
shutil.rmtree(temp_dir)

# Try next:
# 1. Increase NUM_WRITES and see how the ratio changes.
# 2. Think about what "acks=all" means in terms of this buffering model.
# 3. Consider: if your pipeline crashes between write() and flush(), what happens?

---

## 5. Concurrency: Threading with Locks

Threading is the first concurrency tool most engineers reach for. It works well for I/O-bound tasks in Python, but shared mutable state introduces race conditions that can be subtle and intermittent.

### threading module essentials

- `threading.Thread`: create and start a thread
- `threading.Lock`: mutual exclusion -- only one thread can hold it at a time
- `threading.RLock`: reentrant lock -- the same thread can acquire it multiple times without deadlocking itself
- `threading.Event`: simple signaling between threads

### Race condition

A race condition occurs when two or more threads read and modify shared state without synchronization, and the outcome depends on the timing of their execution. The classic example:

1. Thread A reads counter = 10
2. Thread B reads counter = 10
3. Thread A writes counter = 11
4. Thread B writes counter = 11
5. Expected: 12. Actual: 11. One increment was lost.

This is not a theoretical concern. In production, race conditions in shared metrics counters, connection state, and cache entries cause data corruption that is extremely hard to reproduce.

### Lock

A lock provides mutual exclusion. When a thread holds the lock, other threads that try to acquire it will block until it is released. The `with` statement is the correct way to use locks:

```python
with lock:
    # Only one thread executes this block at a time
    shared_state += 1
```

Using `with` guarantees the lock is released even if an exception occurs inside the block.

### RLock (reentrant lock)

An RLock can be acquired multiple times by the same thread without deadlocking. This is useful when a locked function calls another locked function that uses the same lock. A regular Lock would deadlock in that situation.

### Deadlock

Deadlock occurs when two threads each hold a lock the other needs:

1. Thread A holds Lock 1 and waits for Lock 2
2. Thread B holds Lock 2 and waits for Lock 1
3. Neither can proceed. The program hangs forever.

Prevention strategy: **always acquire locks in the same global order**. If every thread acquires Lock 1 before Lock 2, deadlock on those locks is impossible.

### Industry context

Shared counters for pipeline metrics (messages processed, errors seen) need locks or atomic operations. Connection pool state (which connections are in use, which are idle) needs synchronization. The backpressure handler in your energy trading consumer uses threading internally to coordinate between the polling loop and the processing loop.

In [ ]:
import threading
import time

# ============================================================
# Part 1: Demonstrate a race condition
# ============================================================

counter = 0

def increment_unsafe(n: int) -> None:
    """Increment shared counter WITHOUT a lock."""
    global counter
    for _ in range(n):
        # This is NOT atomic: read counter, add 1, write counter
        counter += 1

ITERATIONS = 100_000
NUM_THREADS = 4

counter = 0
threads = [
    threading.Thread(target=increment_unsafe, args=(ITERATIONS,))
    for _ in range(NUM_THREADS)
]
for t in threads:
    t.start()
for t in threads:
    t.join()

expected = ITERATIONS * NUM_THREADS
print(f"Race condition demo:")
print(f"  Expected: {expected:>10}")
print(f"  Actual:   {counter:>10}")
print(f"  Lost:     {expected - counter:>10} increments")

# ============================================================
# Part 2: Fix with a Lock
# ============================================================

counter = 0
counter_lock = threading.Lock()

def increment_safe(n: int) -> None:
    """Increment shared counter WITH a lock."""
    global counter
    for _ in range(n):
        with counter_lock:
            counter += 1

counter = 0
threads = [
    threading.Thread(target=increment_safe, args=(ITERATIONS,))
    for _ in range(NUM_THREADS)
]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"\nWith lock:")
print(f"  Expected: {expected:>10}")
print(f"  Actual:   {counter:>10}")
print(f"  Lost:     {expected - counter:>10} increments")

# ============================================================
# Part 3: RLock -- reentrant locking
# ============================================================

rlock = threading.RLock()

def outer():
    with rlock:
        return inner()

def inner():
    with rlock:  # Same thread re-acquires -- RLock allows this, Lock would deadlock
        return "success"

print(f"\nRLock reentrant call: {outer()}")

# ============================================================
# Part 4: Deadlock demonstration (with timeout to avoid hanging)
# ============================================================

lock_a = threading.Lock()
lock_b = threading.Lock()
deadlock_detected = threading.Event()

def thread_1():
    lock_a.acquire()
    time.sleep(0.01)  # Force interleaving
    got_b = lock_b.acquire(timeout=0.5)  # Timeout prevents actual hang
    if not got_b:
        deadlock_detected.set()
    else:
        lock_b.release()
    lock_a.release()

def thread_2():
    lock_b.acquire()  # Opposite order from thread_1!
    time.sleep(0.01)
    got_a = lock_a.acquire(timeout=0.5)
    if not got_a:
        deadlock_detected.set()
    else:
        lock_a.release()
    lock_b.release()

t1 = threading.Thread(target=thread_1)
t2 = threading.Thread(target=thread_2)
t1.start()
t2.start()
t1.join()
t2.join()

print(f"\nDeadlock detected (via timeout): {deadlock_detected.is_set()}")
print("Fix: always acquire lock_a before lock_b in both threads.")

# Try next:
# 1. Remove the lock from increment_safe and confirm the race returns.
# 2. Change the deadlock example to acquire locks in the same order -- verify no deadlock.
# 3. Increase ITERATIONS and observe whether the race condition magnitude changes.

---

## 6. Concurrency: Multiprocessing

When you need true CPU parallelism in Python, processes are the answer. Each process gets its own Python interpreter and its own GIL, so CPU-bound work actually runs in parallel across cores.

### Why processes bypass the GIL

The GIL is per-interpreter. Since each process has its own interpreter, there is no shared GIL. Multiple processes can execute CPU-bound Python bytecode simultaneously on different cores.

### IPC overhead

The price of isolation: data must be serialized (pickled) to pass between processes. This means:

- sending a large DataFrame between processes copies it entirely
- pickle is not free -- serialization and deserialization take time
- for small tasks, IPC overhead can dominate the actual computation

### multiprocessing.Pool

`Pool` manages a fixed number of worker processes and distributes work to them:

```python
from multiprocessing import Pool
with Pool(4) as pool:
    results = pool.map(compute_heavy, data_chunks)
```

### shared memory

For simple shared state between processes, `multiprocessing.Value` and `multiprocessing.Array` provide shared memory backed by OS primitives. These avoid pickling but are limited to simple C types.

### When NOT to use multiprocessing

- small tasks where IPC overhead dominates
- work that requires complex shared state (use threads or restructure)
- environments where `fork` is unsafe (some macOS/Windows edge cases)

### Windows and notebook caveat

On Windows, `multiprocessing` uses the `spawn` start method, which requires the target function to be importable from the top-level module. This means **`multiprocessing.Pool` does not work reliably in Jupyter notebooks on Windows**. The workaround is to put the worker function and pool code in a `.py` script and run it with `if __name__ == "__main__":` guard.

The code cell below shows the pattern as a script example that you can copy to a `.py` file and run from the command line.

### Industry context

Parallel data transformation on large batches (e.g., computing features across 10M trade records), parallel file parsing, and parallel model inference are common multiprocessing use cases. In your energy trading platform, batch ingestion could use multiprocessing to parse and validate files from multiple sources concurrently.

In [ ]:
# === Multiprocessing script example ===
# Copy this to a file like "mp_benchmark.py" and run: python mp_benchmark.py
# It will NOT work if you run it directly in a Jupyter notebook on Windows.

MULTIPROCESSING_SCRIPT = '''
"""
mp_benchmark.py -- Compare sequential vs multiprocessing for CPU-bound work.
Run: python mp_benchmark.py
"""
import math
import time
from multiprocessing import Pool

def cpu_heavy(n: int) -> float:
    """CPU-bound: sum of square roots."""
    return sum(math.sqrt(i) for i in range(n))

def main():
    WORK_SIZE = 1_000_000
    NUM_TASKS = 8

    # Sequential
    start = time.perf_counter()
    seq_results = [cpu_heavy(WORK_SIZE) for _ in range(NUM_TASKS)]
    seq_time = time.perf_counter() - start

    # Multiprocessing with Pool
    start = time.perf_counter()
    with Pool(4) as pool:
        mp_results = pool.map(cpu_heavy, [WORK_SIZE] * NUM_TASKS)
    mp_time = time.perf_counter() - start

    print(f"Sequential:      {seq_time:.3f}s")
    print(f"Multiprocessing: {mp_time:.3f}s  (4 workers)")
    print(f"Speedup:         {seq_time / mp_time:.1f}x")
    print(f"Results match:   {seq_results == mp_results}")

if __name__ == "__main__":
    main()
'''

print("=== Multiprocessing benchmark script ===")
print("Save the code below to mp_benchmark.py and run from the command line.")
print("It cannot run in a Jupyter notebook on Windows due to spawn limitations.")
print()
print(MULTIPROCESSING_SCRIPT)

# --- What we CAN show in the notebook: shared memory concepts ---
import multiprocessing

# multiprocessing.Value and Array work in notebooks for demonstration
shared_counter = multiprocessing.Value("i", 0)  # 'i' = signed int, initial value 0
shared_array = multiprocessing.Array("d", [1.0, 2.0, 3.0])  # 'd' = double

print(f"Shared counter value: {shared_counter.value}")
print(f"Shared array values:  {list(shared_array)}")
print()
print("These use OS shared memory -- no pickling needed.")
print("But they only support simple C types, not arbitrary Python objects.")

# Try next:
# 1. Save the script above and run it. Compare speedup with different Pool sizes.
# 2. Try Pool(1) vs Pool(8) and observe diminishing returns.
# 3. Reduce WORK_SIZE to 100 and see IPC overhead dominate.

---

## 7. Concurrency: asyncio

asyncio is Python's built-in framework for cooperative multitasking. It runs an event loop on a single thread, dispatching coroutines that yield control while waiting for I/O. This makes it ideal for high-concurrency I/O workloads where you need thousands of concurrent operations without thousands of threads.

### Event loop

The event loop is the scheduler. It maintains a queue of ready coroutines and runs them one at a time. When a coroutine hits an `await` (e.g., waiting for a network response), it yields control back to the loop, which picks up another ready coroutine. This is cooperative: the coroutine must explicitly yield.

### async/await

- `async def` defines a coroutine function
- `await` suspends the coroutine until the awaited operation completes
- a coroutine that never awaits anything will block the entire event loop

### asyncio.gather

`asyncio.gather(*coros)` runs multiple coroutines concurrently and waits for all of them to complete. This is the async equivalent of launching multiple threads.

### When async shines

- thousands of concurrent HTTP API calls (e.g., fan-out ingestion from multiple vendors)
- thousands of concurrent database queries
- websocket connections (hundreds or thousands of simultaneous streams)
- any I/O-bound workload where the bottleneck is waiting, not computing

### When async is wrong

- CPU-bound work: a compute-heavy coroutine blocks the event loop and starves all other coroutines
- simple scripts: the overhead of async/await adds complexity without benefit
- when you need to call blocking libraries that do not have async versions (use `asyncio.to_thread` as a bridge)

### Notebook caveat

Jupyter notebooks already run an event loop. You cannot call `asyncio.run()` inside a notebook. Instead, use `await` directly at the top level, which Jupyter supports natively.

### Industry context

FastAPI is async-native. Your energy trading API uses async endpoints. The ingestion service uses async HTTP clients (httpx/aiohttp) to poll multiple vendor APIs concurrently. The key insight: async lets you overlap thousands of network waits without the memory overhead of thousands of threads.

In [ ]:
import asyncio
import time

# --- Simulate async I/O (e.g., API calls to different vendors) ---

async def fetch_vendor_data(vendor: str, delay: float) -> dict:
    """Simulate fetching data from a vendor API."""
    await asyncio.sleep(delay)  # Non-blocking sleep -- yields to event loop
    return {"vendor": vendor, "records": 100, "latency": delay}

# --- Sequential version for comparison ---
async def sequential_fetch(vendors: list[tuple[str, float]]) -> list[dict]:
    results = []
    for vendor, delay in vendors:
        result = await fetch_vendor_data(vendor, delay)
        results.append(result)
    return results

# --- Concurrent version with gather ---
async def concurrent_fetch(vendors: list[tuple[str, float]]) -> list[dict]:
    coros = [fetch_vendor_data(vendor, delay) for vendor, delay in vendors]
    return await asyncio.gather(*coros)

# Simulate 8 vendor API calls, each taking 0.15-0.25 seconds
vendors = [
    ("finnhub", 0.20), ("entsoe", 0.25), ("binance", 0.15), ("coinbase", 0.18),
    ("kraken", 0.22), ("polygon", 0.17), ("alpaca", 0.19), ("iex", 0.21),
]

# Sequential
start = time.perf_counter()
seq_results = await sequential_fetch(vendors)
seq_time = time.perf_counter() - start

# Concurrent
start = time.perf_counter()
conc_results = await concurrent_fetch(vendors)
conc_time = time.perf_counter() - start

print(f"Sequential ({len(vendors)} vendors): {seq_time:.3f}s")
print(f"Concurrent ({len(vendors)} vendors): {conc_time:.3f}s")
print(f"Speedup: {seq_time / conc_time:.1f}x")
print()
print("Results:")
for r in conc_results:
    print(f"  {r['vendor']:>10}: {r['records']} records, {r['latency']:.2f}s latency")

# --- Demonstrate that blocking calls freeze the event loop ---
async def bad_blocking():
    """This blocks the event loop -- no other coroutine can run."""
    time.sleep(0.5)  # BAD: blocking sleep, not async
    return "done"

async def good_async():
    """This yields to the event loop while waiting."""
    await asyncio.sleep(0.5)  # GOOD: non-blocking
    return "done"

print("\nBlocking vs async sleep:")
start = time.perf_counter()
await asyncio.gather(bad_blocking(), bad_blocking())
bad_time = time.perf_counter() - start

start = time.perf_counter()
await asyncio.gather(good_async(), good_async())
good_time = time.perf_counter() - start

print(f"  2x blocking sleep: {bad_time:.3f}s (sequential -- event loop was frozen)")
print(f"  2x async sleep:    {good_time:.3f}s (concurrent -- event loop was free)")

# Try next:
# 1. Add more vendors and see if concurrent time stays flat.
# 2. Replace asyncio.sleep with time.sleep in fetch_vendor_data and observe the difference.
# 3. Use asyncio.create_task() instead of gather and see how it differs.

---

## 8. Producer-Consumer Pattern

The producer-consumer pattern is one of the most important concurrency patterns in data engineering. Producers generate work items and put them on a queue. Consumers take items from the queue and process them. The queue decouples production rate from consumption rate.

### Why this pattern matters

- **Kafka** is fundamentally a distributed producer-consumer system: producers write to topics, consumers read from partitions
- pipeline stages often follow this pattern: Stage A produces records, Stage B consumes and transforms them
- it naturally handles rate differences: if the producer is faster, work buffers in the queue; if the consumer is faster, it waits

### queue.Queue (thread-safe)

`queue.Queue` is a thread-safe FIFO queue from the standard library. It handles all the locking internally:

- `put(item)`: add an item (blocks if queue is full, if maxsize is set)
- `get()`: remove and return an item (blocks if queue is empty)
- `task_done()`: signal that a previously retrieved item has been processed
- `join()`: block until all items have been processed

### asyncio.Queue

The async equivalent for coroutine-based producer-consumer pipelines. Same API pattern but uses `await`.

### Poison pill pattern

To tell consumers to shut down cleanly, send a sentinel value (often `None`) through the queue. Each consumer, upon receiving the sentinel, stops processing. If there are N consumers, send N sentinels.

### Backpressure

Setting `maxsize` on the queue creates backpressure: when the queue is full, `put()` blocks the producer. This prevents unbounded memory growth when the producer is faster than the consumer. Your energy trading platform's backpressure handler uses exactly this mechanism.

### Industry context

Your Kafka consumer is a consumer in this pattern. The Kafka broker is the queue. The producer writes trade events; the consumer reads, validates, aggregates, and stores them. Internally, the consumer may use another producer-consumer pattern: a polling thread produces raw messages, and processing threads consume them.

In [ ]:
import queue
import random
import threading
import time

# ============================================================
# Complete producer-consumer with multiple producers and consumers
# ============================================================

POISON_PILL = None

def trade_producer(
    producer_id: int,
    work_queue: queue.Queue,
    num_trades: int,
) -> None:
    """Simulate a data source producing trade events."""
    symbols = ["EURUSD", "USDJPY", "GBPUSD", "AUDUSD"]
    for i in range(num_trades):
        trade = {
            "producer": producer_id,
            "symbol": random.choice(symbols),
            "price": round(random.uniform(1.0, 150.0), 4),
            "volume": random.randint(1, 1000),
            "seq": i,
        }
        work_queue.put(trade)
        time.sleep(random.uniform(0.001, 0.005))  # Simulate variable production rate
    print(f"  Producer {producer_id}: finished ({num_trades} trades)")

def trade_consumer(
    consumer_id: int,
    work_queue: queue.Queue,
    results: list,
    results_lock: threading.Lock,
) -> None:
    """Simulate a pipeline stage consuming and processing trade events."""
    local_count = 0
    while True:
        trade = work_queue.get()
        if trade is POISON_PILL:
            work_queue.task_done()
            break
        # "Process" the trade: validate and enrich
        trade["processed_by"] = consumer_id
        trade["validated"] = trade["price"] > 0 and trade["volume"] > 0
        with results_lock:
            results.append(trade)
        local_count += 1
        work_queue.task_done()
    print(f"  Consumer {consumer_id}: processed {local_count} trades")

# --- Configuration ---
NUM_PRODUCERS = 3
NUM_CONSUMERS = 2
TRADES_PER_PRODUCER = 50
MAX_QUEUE_SIZE = 20  # Backpressure: producers block when queue is full

work_queue = queue.Queue(maxsize=MAX_QUEUE_SIZE)
results = []
results_lock = threading.Lock()

print(f"Starting {NUM_PRODUCERS} producers, {NUM_CONSUMERS} consumers")
print(f"Queue maxsize: {MAX_QUEUE_SIZE} (backpressure enabled)")
print()

start = time.perf_counter()

# Start consumers
consumers = [
    threading.Thread(
        target=trade_consumer,
        args=(i, work_queue, results, results_lock),
    )
    for i in range(NUM_CONSUMERS)
]
for c in consumers:
    c.start()

# Start producers
producers = [
    threading.Thread(
        target=trade_producer,
        args=(i, work_queue, TRADES_PER_PRODUCER),
    )
    for i in range(NUM_PRODUCERS)
]
for p in producers:
    p.start()

# Wait for all producers to finish
for p in producers:
    p.join()

# Send poison pills -- one per consumer
for _ in range(NUM_CONSUMERS):
    work_queue.put(POISON_PILL)

# Wait for all items to be processed
work_queue.join()

# Wait for consumer threads to exit
for c in consumers:
    c.join()

elapsed = time.perf_counter() - start

print()
print(f"Total trades produced: {NUM_PRODUCERS * TRADES_PER_PRODUCER}")
print(f"Total trades processed: {len(results)}")
print(f"All validated: {all(r['validated'] for r in results)}")
print(f"Elapsed: {elapsed:.3f}s")

# Show distribution across consumers
from collections import Counter
consumer_dist = Counter(r["processed_by"] for r in results)
print(f"Distribution across consumers: {dict(consumer_dist)}")

# Try next:
# 1. Set MAX_QUEUE_SIZE to 1 and observe how backpressure slows producers.
# 2. Add a third consumer and see how distribution changes.
# 3. Make one consumer slower (add sleep) and watch the other compensate.

---

## 9. Networking: DNS

DNS (Domain Name System) translates human-readable hostnames into IP addresses. It is the first step in every network connection, and when it fails, nothing downstream works.

### What happens during DNS resolution

When your code connects to `api.finnhub.io`:

1. **Local cache check**: the OS checks its local DNS cache. If the hostname was resolved recently, the cached IP is returned immediately.
2. **OS resolver**: the OS sends a query to its configured DNS resolver (typically your ISP's DNS or a public resolver like 8.8.8.8 or 1.1.1.1).
3. **Recursive resolution**: the resolver walks the DNS hierarchy:
   - root nameserver: "I don't know `api.finnhub.io`, but here are the nameservers for `.io`"
   - `.io` TLD nameserver: "Here are the nameservers for `finnhub.io`"
   - `finnhub.io` authoritative nameserver: "The IP for `api.finnhub.io` is 104.26.10.123"
4. **Caching**: the result is cached at every level, governed by the TTL (Time To Live) value set by the domain owner.

### TTL

TTL controls how long DNS results are cached. A low TTL (e.g., 60 seconds) means frequent re-resolution, which enables faster failover but adds latency. A high TTL (e.g., 3600 seconds) reduces DNS traffic but means changes propagate slowly.

### Failure modes

- **"Could not resolve host"**: DNS resolution failed entirely. Check network connectivity, DNS server availability, or typos in the hostname.
- **Stale cache**: the IP changed but your cache still has the old one. This causes connections to the wrong server or to a dead IP.
- **DNS timeout**: the resolver is unreachable or slow. Common in network partitions.

### Industry context

In K8s, service discovery relies on DNS. When you connect to `kafka-broker-0.kafka.default.svc.cluster.local`, K8s CoreDNS resolves that to the pod's cluster IP. If CoreDNS is down or overloaded, every service-to-service connection fails with DNS errors -- even though the target services are healthy. This is why DNS monitoring is critical in K8s clusters.

DNS caching also matters for external API calls. If your ingestion service resolves `api.finnhub.io` once and caches it for an hour, and Finnhub rotates their IPs, your service will fail until the cache expires.

In [ ]:
import socket
import time

# --- DNS resolution in Python (offline-safe examples) ---

hostnames = [
    "localhost",
    "localhost",
    "example.invalid",
]

print("DNS resolution results:")
print(f"  {'Hostname':<20} {'IP Address':<20} {'Time (ms)':<10}")
print(f"  {'-'*20} {'-'*20} {'-'*10}")

for hostname in hostnames:
    start = time.perf_counter()
    try:
        ip = socket.gethostbyname(hostname)
        elapsed_ms = (time.perf_counter() - start) * 1000
        print(f"  {hostname:<20} {ip:<20} {elapsed_ms:<10.2f}")
    except socket.gaierror as exc:
        elapsed_ms = (time.perf_counter() - start) * 1000
        print(f"  {hostname:<20} {'FAILED':<20} {elapsed_ms:<10.2f} ({exc})")

print("
Address info for localhost:")
infos = socket.getaddrinfo("localhost", 443, proto=socket.IPPROTO_TCP)
for family, socktype, proto, canonname, sockaddr in infos[:4]:
    family_name = "IPv4" if family == socket.AF_INET else "IPv6"
    print(f"  {family_name}: {sockaddr[0]}:{sockaddr[1]}")

# Try next:
# 1. Resolve localhost twice and compare timings.
# 2. Why does example.invalid fail immediately?
# 3. What is the difference between gethostbyname and getaddrinfo?


---

## 10. Networking: TCP

TCP (Transmission Control Protocol) provides reliable, ordered delivery of data between two endpoints. Every HTTP request, database query, and Kafka message rides on top of a TCP connection.

### Three-way handshake

Before any data is exchanged, TCP establishes a connection:

```
Client                    Server
  |--- SYN ----------------->|    "I want to connect"
  |<-- SYN-ACK -------------|    "OK, I acknowledge and want to connect too"
  |--- ACK ----------------->|    "Got it, connection established"
```

This takes one round trip (plus a bit). On a 10ms network, that is 10-15ms before any application data flows. On a cross-region connection (50-100ms RTT), the handshake alone takes 50-100ms.

### Connection pooling

Because the handshake is expensive relative to request processing, production systems reuse TCP connections:

- database connection pools (e.g., asyncpg, psycopg pool) keep connections open
- HTTP keep-alive reuses TCP connections for multiple requests
- Kafka clients maintain long-lived connections to brokers

Creating a new TCP connection per request is a common performance anti-pattern.

### Keep-alive

TCP keep-alive sends periodic probes on idle connections to detect if the remote end has died. Without keep-alive, a connection to a crashed server can sit idle indefinitely, looking "connected" from the client side. In K8s, where pods can be killed at any time, keep-alive detects stale connections.

### Failure modes

- **Connection refused** (`ECONNREFUSED`): the target host is reachable, but nothing is listening on that port. Common cause: service is not running, wrong port, firewall ACL.
- **Connection timed out** (`ETIMEDOUT`): no response at all. The host is unreachable, or a firewall is silently dropping packets. Common in network partitions.
- **Connection reset** (`ECONNRESET`): the remote side forcefully closed the connection. Common when a server crashes or a load balancer drops an idle connection.

### Industry context

Your TimescaleDB connection pool keeps TCP connections open so each query does not pay the handshake cost. When a K8s pod restarts, those connections become stale -- the pool needs health checks (keep-alive or explicit ping) to detect and replace dead connections. Kafka clients maintain persistent TCP connections to each broker and reconnect automatically on failure, but during the reconnection window, messages can be delayed.

In [ ]:
import socket
import threading
import time

# --- TCP connection demonstration (self-contained) ---

def start_local_server() -> tuple[socket.socket, int]:
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server.bind(("127.0.0.1", 0))
    server.listen(1)
    port = server.getsockname()[1]

    def serve_once():
        conn, _ = server.accept()
        conn.close()
        server.close()

    threading.Thread(target=serve_once, daemon=True).start()
    return server, port


def try_tcp_connect(host: str, port: int, timeout: float = 2.0) -> dict:
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    sock.settimeout(timeout)
    start = time.perf_counter()
    try:
        sock.connect((host, port))
        elapsed_ms = (time.perf_counter() - start) * 1000
        return {"status": "connected", "time_ms": round(elapsed_ms, 2)}
    except socket.timeout:
        elapsed_ms = (time.perf_counter() - start) * 1000
        return {"status": "timed out", "time_ms": round(elapsed_ms, 2)}
    except ConnectionRefusedError:
        elapsed_ms = (time.perf_counter() - start) * 1000
        return {"status": "connection refused", "time_ms": round(elapsed_ms, 2)}
    except socket.gaierror as exc:
        elapsed_ms = (time.perf_counter() - start) * 1000
        return {"status": f"DNS failed: {exc}", "time_ms": round(elapsed_ms, 2)}
    except OSError as exc:
        elapsed_ms = (time.perf_counter() - start) * 1000
        return {"status": f"error: {exc}", "time_ms": round(elapsed_ms, 2)}
    finally:
        sock.close()


_, open_port = start_local_server()

targets = [
    ("127.0.0.1", open_port, "local server -- should connect"),
    ("127.0.0.1", 19999, "nothing listening -- should refuse"),
    ("example.invalid", 80, "invalid host -- DNS should fail"),
]

print("TCP connection tests:")
for host, port, description in targets:
    result = try_tcp_connect(host, port, timeout=1.0)
    print(f"  {host}:{port:<5} {description:<35} {result['status']} ({result['time_ms']}ms)")

# Try next:
# 1. Start another local server and connect to it.
# 2. Why is connection refused different from DNS failure?
# 3. Why does a successful TCP connection say nothing yet about HTTP correctness?


---

## 11. Networking: HTTP Request Lifecycle

An HTTP request is the highest layer in the network stack that most application engineers interact with daily. Understanding the full chain helps you debug slow or failing API calls.

### The full chain

```
1. DNS Resolution        hostname --> IP address
2. TCP Handshake         SYN / SYN-ACK / ACK
3. TLS Handshake         (if HTTPS) negotiate encryption, verify certificate
4. HTTP Request          send method, path, headers, body
5. HTTP Response         receive status code, headers, body
6. Connection cleanup    close or return to pool for reuse
```

Each step can fail independently. Knowing which step failed tells you where to look.

### Status codes you should know

| Code | Meaning | What to do |
|------|---------|------------|
| 200 | OK | Success |
| 201 | Created | Resource was created (POST) |
| 204 | No Content | Success, no body returned (DELETE) |
| 301 | Moved Permanently | Update the URL; the resource moved |
| 400 | Bad Request | Your request is malformed; fix the client |
| 401 | Unauthorized | Missing or invalid authentication |
| 403 | Forbidden | Authenticated but not authorized |
| 404 | Not Found | Resource does not exist at this path |
| 429 | Too Many Requests | Rate limited; back off and retry (check `Retry-After` header) |
| 500 | Internal Server Error | Server bug; retry with backoff |
| 502 | Bad Gateway | Proxy/LB could not reach the upstream; upstream may be down |
| 503 | Service Unavailable | Server overloaded or in maintenance; retry with backoff |

### Headers that matter

- `Content-Type`: tells the client/server what format the body is in (e.g., `application/json`)
- `Authorization`: carries credentials (Bearer tokens, API keys)
- `Retry-After`: tells you how long to wait before retrying a 429 or 503
- `Cache-Control`: controls caching behavior for proxies and clients
- `X-Request-ID` / `X-Correlation-ID`: trace a request through distributed systems

### Retry behavior

A well-behaved client:

1. Retries on 429, 500, 502, 503 (transient errors)
2. Does NOT retry on 400, 401, 403, 404 (client errors -- retrying will not help)
3. Uses exponential backoff with jitter to avoid thundering herd
4. Respects `Retry-After` headers when present

### Industry context

When your energy trading ingestion service calls the Finnhub API and gets a 429, the circuit breaker in your resilience layer should back off. When it gets a 502, the Kafka consumer should retry because the upstream might just be restarting. The difference between these status codes drives the retry logic in your `src/ingestion/` adapters.

In [ ]:
import http.server
import socketserver
import threading
import urllib.error
import urllib.request
import time

# --- HTTP request lifecycle demonstration (self-contained local server) ---

class DemoHandler(http.server.BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == "/ok":
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(b'{"status": "ok"}')
        elif self.path == "/missing":
            self.send_response(404)
            self.end_headers()
        else:
            self.send_response(500)
            self.end_headers()

    def log_message(self, format, *args):
        return


def start_http_server() -> tuple[socketserver.TCPServer, int]:
    server = socketserver.TCPServer(("127.0.0.1", 0), DemoHandler)
    port = server.server_address[1]
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, port


def http_get(url: str, timeout: float = 5.0) -> dict:
    result = {"url": url}
    start = time.perf_counter()
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Python-Notebook/1.0"})
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            elapsed_ms = (time.perf_counter() - start) * 1000
            result["status"] = resp.status
            result["headers"] = dict(resp.headers)
            result["body_size"] = len(resp.read())
            result["content_type"] = resp.headers.get("Content-Type", "unknown")
            result["time_ms"] = round(elapsed_ms, 2)
    except urllib.error.HTTPError as exc:
        elapsed_ms = (time.perf_counter() - start) * 1000
        result["status"] = exc.code
        result["error"] = exc.reason
        result["time_ms"] = round(elapsed_ms, 2)
    return result


server, port = start_http_server()
base = f"http://127.0.0.1:{port}"
test_urls = [f"{base}/ok", f"{base}/missing", f"{base}/error"]

print("HTTP request lifecycle tests:")
for url in test_urls:
    result = http_get(url)
    print(f"  {url}")
    print(f"    Status: {result.get('status')}  Time: {result.get('time_ms')}ms  {result.get('error', '')}")
    print()

server.shutdown()
server.server_close()


def classify_status(code: int) -> str:
    if 200 <= code < 300:
        return "success -- no retry needed"
    elif code in (429, 500, 502, 503):
        return "RETRY with exponential backoff"
    elif 400 <= code < 500:
        return "client error -- do NOT retry (fix the request)"
    else:
        return "unexpected -- investigate"

print("Status code retry decisions:")
for code in [200, 201, 301, 400, 401, 404, 429, 500, 502, 503]:
    print(f"  {code}: {classify_status(code)}")

# Try next:
# 1. Add a local /slow endpoint and enforce a short timeout.
# 2. Why is a 404 usually not retried?
# 3. Why is a 500 often retried with backoff?


---

## 12. Networking: TLS

TLS (Transport Layer Security) encrypts the communication channel and verifies the server's identity. Every HTTPS connection uses TLS. Understanding it at a conceptual level helps you debug certificate errors and reason about security in your infrastructure.

### What TLS does

1. **Encrypts data in transit**: prevents anyone on the network path from reading the traffic
2. **Authenticates the server**: the client verifies the server is who it claims to be using certificates
3. **Ensures integrity**: detects any tampering with the data in transit

### The TLS handshake (simplified)

After the TCP handshake completes:

```
Client                              Server
  |--- ClientHello ----------------->|   supported TLS versions, cipher suites
  |<-- ServerHello + Certificate ----|   chosen cipher, server's certificate
  |    (client verifies certificate)
  |--- Key Exchange + Finished ----->|   shared secret established
  |<-- Finished ---------------------|   encrypted channel ready
```

This adds another round trip (or two) before any application data flows. TLS 1.3 reduced this to one round trip, and supports 0-RTT resumption for repeat connections.

### Certificate chain

The server presents a certificate chain:

- **Server certificate**: "I am api.finnhub.io, and here is my public key"
- **Intermediate CA**: "I vouched for api.finnhub.io, and a root CA vouched for me"
- **Root CA**: "I am a trusted root, pre-installed in your OS/browser trust store"

The client walks the chain from server cert to root CA. If any link is broken, the verification fails.

### Common TLS failures

- **Certificate expired**: the server certificate has a validity period; if it expires, all clients reject the connection
- **Wrong hostname**: the certificate was issued for `example.com` but the client connected to `api.example.com`
- **Self-signed certificate**: the certificate was not issued by a trusted CA; common in development environments
- **Incomplete chain**: the server did not include the intermediate CA certificate

### Mutual TLS (mTLS)

In standard TLS, only the server proves its identity. In mutual TLS, the client also presents a certificate to the server. This is used in:

- K8s service meshes (Istio, Linkerd) for pod-to-pod authentication
- zero-trust architectures where every service must prove its identity
- financial systems where both parties must be authenticated

### Industry context

Certificate expiry is a leading cause of production outages. When your K8s cluster's internal certificates expire, pod-to-pod communication fails. When Istio's mTLS certificates rotate, there is a brief window where connections may fail. Automated certificate rotation (cert-manager in K8s) is essential for reliability.

In [ ]:
import datetime

# --- TLS certificate structure demonstration (offline-safe) ---

fake_cert = {
    "subject": ((('commonName', 'api.example.com'),),),
    "issuer": ((('organizationName', 'Example Intermediate CA'),),),
    "notBefore": "Apr 01 00:00:00 2026 GMT",
    "notAfter": "Jun 30 23:59:59 2026 GMT",
    "subjectAltName": (("DNS", "api.example.com"), ("DNS", "example.com")),
}

not_before = datetime.datetime.strptime(fake_cert["notBefore"], "%b %d %H:%M:%S %Y %Z")
not_after = datetime.datetime.strptime(fake_cert["notAfter"], "%b %d %H:%M:%S %Y %Z")
days_until_expiry = (not_after - datetime.datetime.utcnow()).days

cert_info = {
    "subject": dict(x[0] for x in fake_cert["subject"]),
    "issuer": dict(x[0] for x in fake_cert["issuer"]),
    "not_before": not_before.isoformat(),
    "not_after": not_after.isoformat(),
    "days_until_expiry": days_until_expiry,
    "san_count": len(fake_cert.get("subjectAltName", [])),
}

cert_info

# Hostname mismatch example:
# certificate SANs = api.example.com, example.com
# requested hostname = wrong.example.com
# result = certificate verification failure


---

## 13. Debugging Tools Cheat Sheet

These are the Linux tools you will use most when debugging production systems. You do not need to memorize every flag, but you should know what each tool does and when to reach for it.

### Process inspection

| Command | What it shows | When to use |
|---------|--------------|-------------|
| `ps aux` | All running processes with CPU/memory usage | "What is running on this machine?" |
| `ps aux \| grep python` | Filter for Python processes | "Is my service running?" |
| `top` / `htop` | Live process monitoring, sorted by resource usage | "What is consuming CPU/memory right now?" |
| `kill -0 PID` | Check if a process is alive (no signal sent) | "Is this PID still running?" |

### File descriptor and I/O inspection

| Command | What it shows | When to use |
|---------|--------------|-------------|
| `lsof -p PID` | All open files, sockets, pipes for a process | "Why is this process using so many FDs?" |
| `lsof -i :8080` | What process is listening on port 8080 | "What is using this port?" |
| `ls -la /proc/PID/fd/` | File descriptor directory (Linux only) | "What FDs does this process have open?" |
| `cat /proc/PID/status` | Process memory, threads, state | "How much memory is this process using?" |

### Network inspection

| Command | What it shows | When to use |
|---------|--------------|-------------|
| `ss -tlnp` | Listening TCP sockets with process info | "What ports are open on this machine?" |
| `ss -tnp` | Established TCP connections | "What is connected to what?" |
| `curl -v URL` | Verbose HTTP request showing DNS, TCP, TLS, HTTP | "Debug the full request lifecycle" |
| `dig hostname` | DNS resolution details | "What IP does this hostname resolve to?" |
| `nslookup hostname` | Simple DNS lookup | Quick hostname-to-IP check |

### System call tracing

| Command | What it shows | When to use |
|---------|--------------|-------------|
| `strace -p PID` | Live system call trace | "What is this process doing at the OS level?" |
| `strace -c -p PID` | Summarize system call counts and times | "Where is this process spending time in syscalls?" |
| `strace -e trace=network -p PID` | Only network-related syscalls | "What network calls is this process making?" |

### Practical debugging workflow

1. **Service not responding?** Check `ps aux | grep service_name` to see if it is running.
2. **High memory?** Check `top` or `cat /proc/PID/status` for VmRSS (resident memory).
3. **"Too many open files"?** Check `lsof -p PID | wc -l` for FD count.
4. **Port conflict?** Check `ss -tlnp | grep PORT` to see what is listening.
5. **Request failing?** Use `curl -v URL` to see which layer (DNS/TCP/TLS/HTTP) fails.
6. **Process hanging?** Use `strace -p PID` to see what syscall it is stuck on.

---

## 14. Network Failure Diagnosis Guide

When a network request fails, the symptoms tell you which layer broke. This decision tree helps you narrow down the problem quickly.

### Decision tree

```
Request failed
  |
  +-- "Could not resolve host"
  |     --> DNS layer failure
  |     --> Check: is DNS server reachable? Is hostname correct? Is CoreDNS running?
  |
  +-- "Connection refused"
  |     --> TCP layer: host is reachable, but nothing is listening on that port
  |     --> Check: is the service running? Is the port correct? Is there a firewall ACL?
  |
  +-- "Connection timed out"
  |     --> TCP layer: no response at all
  |     --> Check: is the host reachable? Is a firewall dropping packets? Network partition?
  |
  +-- "Connection reset"
  |     --> TCP layer: remote side forcefully closed
  |     --> Check: did the server crash? Did a load balancer drop an idle connection?
  |
  +-- "Certificate verify failed"
  |     --> TLS layer: TCP succeeded but TLS handshake failed
  |     --> Check: is the cert expired? Wrong hostname? Self-signed? Missing intermediate CA?
  |
  +-- "SSL: CERTIFICATE_VERIFY_FAILED"
  |     --> TLS layer (Python-specific error message for the same issue)
  |
  +-- Got HTTP response but wrong status
        --> Application layer
        +-- 4xx: client error (bad request, auth failure, not found)
        +-- 5xx: server error (retry with backoff)
        +-- 502/503: upstream/proxy issue (check upstream health)
```

### Interpreting `curl -v` output

`curl -v` shows every layer of the request lifecycle. Here is what to look for:

```bash
$ curl -v https://api.example.com/data

* Trying 93.184.216.34:443...          # <-- DNS resolved, TCP connecting
* Connected to api.example.com          # <-- TCP handshake succeeded
* SSL connection using TLSv1.3          # <-- TLS handshake succeeded
> GET /data HTTP/2                      # <-- HTTP request sent
< HTTP/2 200                            # <-- HTTP response received
```

If it fails at "Trying..." -- DNS or TCP issue.
If it fails at "SSL connection" -- TLS issue.
If you get an HTTP response but the wrong status -- application issue.

In [ ]:
# --- Network failure diagnosis simulator (message-driven, no external network needed) ---

network_failure_map = {
    "connection refused": {
        "layer": "TCP",
        "meaning": "Target host reachable, but nothing accepting connections on that port.",
        "debug": "Check: is the service running? Correct port? Firewall ACL?",
        "tools": "ss -tlnp | grep PORT, curl -v, telnet host port",
    },
    "connection timed out": {
        "layer": "TCP",
        "meaning": "No response at all from target. Host unreachable or firewall dropping packets.",
        "debug": "Check: network connectivity, routing, firewall rules, security groups.",
        "tools": "ping host, traceroute host, curl -v --connect-timeout 5",
    },
    "could not resolve host": {
        "layer": "DNS",
        "meaning": "DNS resolution failed before any TCP connection was attempted.",
        "debug": "Check: hostname spelling, DNS server reachability, CoreDNS health.",
        "tools": "dig hostname, nslookup hostname, cat /etc/resolv.conf",
    },
    "certificate verify failed": {
        "layer": "TLS",
        "meaning": "TCP connected but TLS handshake failed. Certificate issue.",
        "debug": "Check: cert expiry, hostname match, intermediate CA, self-signed.",
        "tools": "openssl s_client -connect host:443, curl -v https://host",
    },
    "502 bad gateway": {
        "layer": "HTTP/Application",
        "meaning": "Proxy or load balancer could not reach the upstream service.",
        "debug": "Check: is the upstream service running? Health check passing?",
        "tools": "kubectl logs, kubectl get pods, curl -v upstream-url",
    },
    "429 too many requests": {
        "layer": "HTTP/Application",
        "meaning": "Rate limited by the server. Too many requests in a time window.",
        "debug": "Check Retry-After header. Implement exponential backoff.",
        "tools": "curl -v (check response headers for Retry-After)",
    },
}

def diagnose_failure(error_message: str) -> None:
    error_lower = error_message.lower()
    matched = False
    for pattern, info in network_failure_map.items():
        if pattern in error_lower:
            print(f"Error: '{error_message}'")
            print(f"  Layer:     {info['layer']}")
            print(f"  Meaning:   {info['meaning']}")
            print(f"  Debug:     {info['debug']}")
            print(f"  Tools:     {info['tools']}")
            print()
            matched = True
            break
    if not matched:
        print(f"Error: '{error_message}'")
        print("  No pattern matched. Check the full stack trace for more context.")
        print()

print("=== Network Failure Diagnosis ===
")
for message in [
    "Could not resolve host: example.invalid",
    "Connection refused: [Errno 111] Connection refused",
    "certificate verify failed: hostname mismatch",
    "HTTP 404 Not Found",
    "HTTP 502 Bad Gateway",
]:
    diagnose_failure(message)

# Try next:
# 1. Add a rule for 'connection reset'.
# 2. Why is a certificate error not the same as a TCP error?
# 3. Why is a 404 handled differently from a 502?


---

## 15. Mini Lab

Work through these tasks using what you have learned in this notebook. Do not search the web first -- use the concepts and code patterns from above.

### Systems tasks

1. Write a function that recursively builds a nested dictionary 500 levels deep. Predict what will happen, then run it.
2. Write a function that intentionally leaks 100 file descriptors. Show the FD count before and after, then fix it.
3. Write a small buffered-write benchmark: compare writing 10,000 lines with default buffering vs flush-per-line. Report the time difference.

### Concurrency tasks

4. Create a shared list protected by a lock. Have 4 threads each append 1,000 items. Verify the final list length is exactly 4,000.
5. Build an async pipeline: 3 async "fetchers" that each simulate 5 API calls with random delays, all running concurrently via `gather`. Report total time vs estimated sequential time.
6. Extend the producer-consumer example: add a "transformer" stage between producer and consumer (producer --> transformer queue --> consumer). Use two queues.

### Networking tasks

7. Resolve 5 different hostnames and time each resolution. Which one is fastest? Why? (Hint: caching.)
8. Attempt TCP connections to 3 different host:port combinations -- one that should succeed, one that should refuse, and one that should time out. Classify each result.
9. Inspect the TLS certificate of a website you use daily. When does it expire? Who issued it?

### Diagnosis task

10. Given this `curl -v` output, identify which layer failed and what the fix is:

```
* Trying 10.0.5.42:8080...
* connect to 10.0.5.42 port 8080 failed: Connection refused
* Failed to connect to api-service.default.svc.cluster.local port 8080: Connection refused
```

---

## 16. Exit Criteria

Do not move on until you can say yes to these:

- I can explain common process, memory, and file-descriptor failures.
- I can explain the difference between processes, threads, and coroutines, and when to use each.
- I can explain how the GIL interacts with each concurrency model.
- I can write a race condition, explain why it happens, and fix it with a lock.
- I can implement a producer-consumer pipeline with proper shutdown.
- I can reason about a basic network request path from DNS to HTTP response.
- I can use basic Linux tooling to inspect process, file, and network state.
- I can debug at least one level below the application code.
- I can trace a network request from DNS to HTTP response and explain each layer.
- I can diagnose which network layer failed given symptoms.
- I can explain why buffered I/O can hide latency or durability issues.
- I can explain what file descriptors are and why "too many open files" happens.

---

## 17. References

- Python Standard Library: `threading`
  https://docs.python.org/3/library/threading.html
- Python Standard Library: `multiprocessing`
  https://docs.python.org/3/library/multiprocessing.html
- Python Standard Library: `asyncio`
  https://docs.python.org/3/library/asyncio.html
- Python Standard Library: `queue`
  https://docs.python.org/3/library/queue.html
- Python Standard Library: `socket`
  https://docs.python.org/3/library/socket.html
- Python Standard Library: `ssl`
  https://docs.python.org/3/library/ssl.html
- Python Standard Library: `os` (fsync, file descriptors)
  https://docs.python.org/3/library/os.html
- Python Standard Library: `sys` (getrecursionlimit, getsizeof)
  https://docs.python.org/3/library/sys.html
- Linux man pages: `ps(1)`, `top(1)`, `lsof(8)`, `ss(8)`, `strace(1)`, `curl(1)`
- The Linux Programming Interface (Kerrisk) -- chapters on processes, memory, file I/O, sockets
- Beej's Guide to Network Programming
  https://beej.us/guide/bgnet/
- High Performance Browser Networking (Grigorik) -- DNS, TCP, TLS, HTTP
  https://hpbn.co/

## Interview Question Bank

Use these after you finish the notebook. Keep the answers concrete.

- What is the difference between a process, a thread, and a coroutine?
- How does the GIL affect thread-based Python programs?
- What is the practical difference between stack memory and heap memory?
- What causes a file-descriptor leak, and how do you prevent one?
- What is the difference between `flush()` and `fsync()`?
- Walk through a request from DNS lookup to TCP handshake to TLS to HTTP response.
- How do you classify `connection refused`, `connection timed out`, and certificate-verification failures?
- Why can a system be concurrency-correct and still be operationally fragile?
